In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('data-cleaning-1').getOrCreate()

In [3]:
spark

In [4]:
df_pyspark = spark.read.csv('marketing_campaign_data_messy.csv', header=True, inferSchema=True)
rows,cols = df_pyspark.count(), len(df_pyspark.columns)
print(f'Loading succsessful. Successfully loaded {rows} rows and {cols} columns')
df_pyspark.show()

Loading succsessful. Successfully loaded 2020 rows and 12 columns
+-------------+--------------------+-------------------+----------+----------+-----------+-------+-------+-----------+------+------+------------+
| Campaign_ID |       Campaign_Name|         Start_Date|  End_Date|   Channel|Impressions|Clicks |  Spend|Conversions|Active|Clicks|Campaign_Tag|
+-------------+--------------------+-------------------+----------+----------+-----------+-------+-------+-----------+------+------+------------+
|    CMP-00001| Q4_Summer_CMP-00001|2023-11-24 00:00:00|2023-12-13|    TikTok|      16795|    197|$102.82|       20.0|     Y|  NULL|          TI|
|    CMP-00002| Q1_Launch_CMP-00002|2023-05-06 00:00:00|2023-05-12|  Facebook|       1860|     30|  24.33|        1.0|     0|  NULL|          FA|
|    CMP-00003| Q3_Winter_CMP-00003|2023-12-13 00:00:00|2023-12-20|     Email|      77820|    843|1323.39|       51.0|    No|  NULL|          EM|
|    CMP-00004|Q1_BlackFriday_CM...|         2023-10-30|20

Fixing the column names

In [5]:
df_pyspark.columns

[' Campaign_ID ',
 'Campaign_Name',
 'Start_Date',
 'End_Date',
 'Channel',
 'Impressions',
 'Clicks ',
 'Spend',
 'Conversions',
 'Active',
 'Clicks',
 'Campaign_Tag']

In [6]:
for c in df_pyspark.columns:
    df_pyspark = df_pyspark.withColumnRenamed(c, c.strip().lower().replace(' ', '_'))

print('Columns renamed successfully')
print(df_pyspark.columns)
df_pyspark.show(5)

Columns renamed successfully
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']
+-----------+--------------------+-------------------+----------+--------+-----------+------+-------+-----------+------+------+------------+
|campaign_id|       campaign_name|         start_date|  end_date| channel|impressions|clicks|  spend|conversions|active|clicks|campaign_tag|
+-----------+--------------------+-------------------+----------+--------+-----------+------+-------+-----------+------+------+------------+
|  CMP-00001| Q4_Summer_CMP-00001|2023-11-24 00:00:00|2023-12-13|  TikTok|      16795|   197|$102.82|       20.0|     Y|  NULL|          TI|
|  CMP-00002| Q1_Launch_CMP-00002|2023-05-06 00:00:00|2023-05-12|Facebook|       1860|    30|  24.33|        1.0|     0|  NULL|          FA|
|  CMP-00003| Q3_Winter_CMP-00003|2023-12-13 00:00:00|2023-12-20|   Email|      77820|   843|1323.39|       51.

Fix spend column

In [7]:
df_pyspark.printSchema()

root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- spend: string (nullable = true)
 |-- conversions: double (nullable = true)
 |-- active: string (nullable = true)
 |-- clicks: double (nullable = true)
 |-- campaign_tag: string (nullable = true)



In [8]:
from pyspark.sql.functions import col, regexp_replace, trim
from pyspark.sql.types import DoubleType

In [9]:
df_pyspark = df_pyspark.withColumn('fixed_spend', regexp_replace(col('spend'), '[^0-9.]', ''))
print('Fixed spend column created successfully')
df_pyspark.select('spend', 'fixed_spend').show(5)

Fixed spend column created successfully
+-------+-----------+
|  spend|fixed_spend|
+-------+-----------+
|$102.82|     102.82|
|  24.33|      24.33|
|1323.39|    1323.39|
|2180.38|    2180.38|
| 252.44|     252.44|
+-------+-----------+
only showing top 5 rows



In [10]:
df_pyspark = df_pyspark.withColumn('fixed_spend', col('fixed_spend').cast(DoubleType()))
df_pyspark.select('spend', 'fixed_spend').printSchema()

root
 |-- spend: string (nullable = true)
 |-- fixed_spend: double (nullable = true)



Fix start data and end date datatype

In [13]:
df_pyspark.select('start_date', 'end_date').printSchema()

root
 |-- start_date: string (nullable = true)
 |-- end_date: date (nullable = true)



In [15]:
from pyspark.sql.types import DateType

In [17]:
df_pyspark = df_pyspark.withColumns({
    'start_date': col('start_date').cast(DateType()),
    'end_date': col('end_date').cast(DateType())
})
df_pyspark.select('start_date', 'end_date').printSchema()
df_pyspark.show()

root
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)

+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|campaign_id|       campaign_name|start_date|  end_date|   channel|impressions|clicks|  spend|conversions|active|clicks|campaign_tag|fixed_spend|
+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|  CMP-00001| Q4_Summer_CMP-00001|2023-11-24|2023-12-13|    TikTok|      16795|   197|$102.82|       20.0|     Y|  NULL|          TI|     102.82|
|  CMP-00002| Q1_Launch_CMP-00002|2023-05-06|2023-05-12|  Facebook|       1860|    30|  24.33|        1.0|     0|  NULL|          FA|      24.33|
|  CMP-00003| Q3_Winter_CMP-00003|2023-12-13|2023-12-20|     Email|      77820|   843|1323.39|       51.0|    No|  NULL|          EM|    1323.39|
|  CMP-00004|Q1_BlackFriday_CM...|2023-1

Fix channel data

In [18]:
df_pyspark.select('channel').distinct().show()

+----------+
|   channel|
+----------+
|   Tik_Tok|
|     Gogle|
|    TikTok|
|     Email|
| Instagram|
|   Facebok|
|Insta_gram|
|       N/A|
|Google Ads|
|    E-mail|
|  Facebook|
+----------+



In [22]:
channel_mapping = {
    'E-mail': 'Email',
    'Facebok': 'Facebook',
    'Google': 'Google Ads',
    'Insta_gram': 'Instagram',
    'Tik_Tok': 'TikTok'
}

df_pyspark = df_pyspark.replace(channel_mapping, subset=['channel'])
print('Channel names standardized successfully')
df_pyspark.select('channel').distinct().show()
df_pyspark.show()


Channel names standardized successfully
+----------+
|   channel|
+----------+
|     Gogle|
|    TikTok|
|     Email|
| Instagram|
|       N/A|
|Google Ads|
|  Facebook|
+----------+

+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|campaign_id|       campaign_name|start_date|  end_date|   channel|impressions|clicks|  spend|conversions|active|clicks|campaign_tag|fixed_spend|
+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|  CMP-00001| Q4_Summer_CMP-00001|2023-11-24|2023-12-13|    TikTok|      16795|   197|$102.82|       20.0|     Y|  NULL|          TI|     102.82|
|  CMP-00002| Q1_Launch_CMP-00002|2023-05-06|2023-05-12|  Facebook|       1860|    30|  24.33|        1.0|     0|  NULL|          FA|      24.33|
|  CMP-00003| Q3_Winter_CMP-00003|2023-12-13|2023-12-20|     Email|      77820|   843|

Fix active column values

In [23]:
df_pyspark.select('active').distinct().show()

+------+
|active|
+------+
|     0|
| False|
|     Y|
|    No|
|   Yes|
|     1|
|  True|
+------+



In [26]:
active_mapping = {
    'Yes': 'True',
    'No': 'False',
    '0': 'False',
    '1': 'True',
    'Y': 'True',
    'N': 'False'
}

df_pyspark = df_pyspark.replace(active_mapping, subset=['active'])
print('Active column standardized successfully')
df_pyspark.select('active').distinct().show()
df_pyspark.show()

Active column standardized successfully
+------+
|active|
+------+
| False|
|  True|
+------+

+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|campaign_id|       campaign_name|start_date|  end_date|   channel|impressions|clicks|  spend|conversions|active|clicks|campaign_tag|fixed_spend|
+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|  CMP-00001| Q4_Summer_CMP-00001|2023-11-24|2023-12-13|    TikTok|      16795|   197|$102.82|       20.0|  True|  NULL|          TI|     102.82|
|  CMP-00002| Q1_Launch_CMP-00002|2023-05-06|2023-05-12|  Facebook|       1860|    30|  24.33|        1.0| False|  NULL|          FA|      24.33|
|  CMP-00003| Q3_Winter_CMP-00003|2023-12-13|2023-12-20|     Email|      77820|   843|1323.39|       51.0| False|  NULL|          EM|    1323.39|
|  CMP-00004|Q1_BlackFriday_C

In [31]:
from pyspark.sql.types import BooleanType
df_pyspark = df_pyspark.withColumn('active', col('active').cast(BooleanType()))
df_pyspark.select('active').printSchema()

df_pyspark.show()

root
 |-- active: boolean (nullable = true)

+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|campaign_id|       campaign_name|start_date|  end_date|   channel|impressions|clicks|  spend|conversions|active|clicks|campaign_tag|fixed_spend|
+-----------+--------------------+----------+----------+----------+-----------+------+-------+-----------+------+------+------------+-----------+
|  CMP-00001| Q4_Summer_CMP-00001|2023-11-24|2023-12-13|    TikTok|      16795|   197|$102.82|       20.0|  true|  NULL|          TI|     102.82|
|  CMP-00002| Q1_Launch_CMP-00002|2023-05-06|2023-05-12|  Facebook|       1860|    30|  24.33|        1.0| false|  NULL|          FA|      24.33|
|  CMP-00003| Q3_Winter_CMP-00003|2023-12-13|2023-12-20|     Email|      77820|   843|1323.39|       51.0| false|  NULL|          EM|    1323.39|
|  CMP-00004|Q1_BlackFriday_CM...|2023-10-30|2023-11-03|    TikTok|      55886|